In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [ ]:
# 数据批次与上下文
batch_size = 16
block_size = 32

# 训练与评估
max_iters = 500       # 更新参数500次
eval_interval = 100   # 每隔100步评估一次
eval_iters = 20       # 每次评估，每个数据集抽20批取平均
learning_rate = 3e-4  # 学习率0.0003

# 模型结构
n_embd = 64
n_head = 4
n_layer = 2
dropout = 0.2

# 运行设备
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print("运行设备：", device)
print("每个注意力头的维度：", n_embd // n_head)

运行设备： cpu
每个注意力头的维度： 16


In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("字符总数", len(text))
print(text[:200])

字符总数 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)

print("词表大小：", vocab_size)
print("所有字符", "".join(chars))

词表大小： 65
所有字符 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s]
decode = lambda l : ''.join(itos[i] for i in l)

sample=text[:20]
token_ids=encode(sample)

print("原文：", repr(sample))
print("编号：", token_ids)
print("还原：", repr(decode(token_ids)))
print("是否一致：", decode(token_ids) == sample)

原文： 'First Citizen:\nBefor'
编号： [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56]
还原： 'First Citizen:\nBefor'
是否一致： True


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("完整数据形状：", data.shape)
print("数据类型：", data.dtype)
print("训练集长度：", len(train_data))
print("验证集长度：", len(val_data))

# Tensor 转回列表后，可以用之前的 decode 还原
print(decode(data[:100].tolist()))

完整数据形状： torch.Size([1115394])
数据类型： torch.int64
训练集长度： 1003854
验证集长度： 111540
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')

print("输入形状：", xb.shape)
print("目标形状：", yb.shape)

print("第一段输入：", xb[0])
print("第一段目标：", yb[0])

print("输入文字：", repr(decode(xb[0].tolist())))
print("目标文字：", repr(decode(yb[0].tolist())))

输入形状： torch.Size([16, 32])
目标形状： torch.Size([16, 32])
第一段输入： tensor([58, 53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,
         1, 61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1])
第一段目标： tensor([53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,  1,
        61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1, 57])
输入文字： "to corrupt a man's wife is\nwhen "
目标文字： "o corrupt a man's wife is\nwhen s"


In [ ]:
class Head(nn.Module):
    """一个自注意力头"""

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

In [ ]:
head = Head(n_embd // n_head)
print(head)
print("掩码形状：", head.tril.shape)
print(head.tril[:4, :4])

Head(
  (key): Linear(in_features=64, out_features=16, bias=False)
  (query): Linear(in_features=64, out_features=16, bias=False)
  (value): Linear(in_features=64, out_features=16, bias=False)
  (dropout): Dropout(p=0.2, inplace=False)
)
掩码形状： torch.Size([32, 32])
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [ ]:
# 重新创建对象，让它使用刚刚补全的类定义
head = Head(n_embd // n_head).to(device)

# 暂时模拟一批特征，后面再换成真正的嵌入输出
x = torch.randn(batch_size, block_size, n_embd, device=device)
out = head(x)

print("输入形状：", x.shape)
print("输出形状：", out.shape)

输入形状： torch.Size([16, 32, 64])
输出形状： torch.Size([16, 32, 16])


In [ ]:
class MultiHeadAttention(nn.Module):
    """并行运行的多个自注意力头"""

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
multi_head = MultiHeadAttention(n_head, n_embd // n_head).to(device)

x = torch.randn(batch_size, block_size, n_embd, device=device)
out = multi_head(x)

print("输入形状：", x.shape)
print("输出形状：", out.shape)

输入形状： torch.Size([16, 32, 64])
输出形状： torch.Size([16, 32, 64])


In [ ]:
class FeedFoward(nn.Module):
    """一个简单的线性层，后接非线性变换"""

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
ffwd = FeedFoward(n_embd).to(device)

x = torch.randn(batch_size, block_size, n_embd, device=device)
out = ffwd(x)

print("输入形状：", x.shape)
print("输出形状：", out.shape)

输入形状： torch.Size([16, 32, 64])
输出形状： torch.Size([16, 32, 64])


In [ ]:
class Block(nn.Module):
    """Transformer 块：先交换信息，再进行计算"""

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
block = Block(n_embd, n_head).to(device)

x = torch.randn(batch_size, block_size, n_embd, device=device)
out = block(x)

print("输入形状：", x.shape)
print("输出形状：", out.shape)

输入形状： torch.Size([16, 32, 64])
输出形状： torch.Size([16, 32, 64])


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table((torch.arange(T, device=device)))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [ ]:
test_model = GPTLanguageModel()

print("字符嵌入表：", test_model.token_embedding_table.weight.shape)
print("位置嵌入表：", test_model.position_embedding_table.weight.shape)
print("Block 数量：", len(test_model.blocks))

字符嵌入表： torch.Size([65, 64])
位置嵌入表： torch.Size([32, 64])
Block 数量： 2


In [ ]:
# 重新创建模型，使用刚补全的类定义
model = GPTLanguageModel()
m = model.to(device)

xb, yb = get_batch('train')

# 有目标：计算损失，logits 会展平
logits, loss = model(xb, yb)

print("训练时logits形状：", logits.shape)
print("loss：", loss.item())

# 无目标：保留批次与位置两个维度
with torch.no_grad():
    logits, loss = model(xb)

print("无目标时logits形状：", logits.shape)
print("loss：", loss)

训练时 logits 形状： torch.Size([512, 65])
loss： 4.172262668609619
无目标时 logits 形状： torch.Size([16, 32, 65])
loss： None


In [ ]:
model = GPTLanguageModel()
m = model.to(device)

# 关闭 Dropout，生成时不记录梯度
m.eval()

context = torch.zeros((1, 1), dtype=torch.long, device=device)

with torch.no_grad():
    result = m.generate(context, max_new_tokens=20)

print("生成结果形状：", result.shape)
print("生成文字：", repr(decode(result[0].tolist())))

# 恢复训练模式
m.train()

生成结果形状： torch.Size([1, 21])
生成文字： '\n:IZjbLYFAFzDEO-HHQOW'


GPTLanguageModel(
  (token_embedding_table): Embedding(65, 64)
  (position_embedding_table): Embedding(32, 64)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=64, out_features=16, bias=False)
            (query): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedFoward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=Tr

In [ ]:
@torch.no_grad
def estimate_loss():
    out = {}
    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [ ]:
losses = estimate_loss()

print("训练集 loss：", losses['train'].item())
print("验证集 loss：", losses['val'].item())

训练集 loss： 4.179736137390137
验证集 loss： 4.1820173263549805


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(
            f"step {iter}:"
            f"train loss {losses['train']:.4f},"
            f"val loss {losses['val']:.4f}"
        )
    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0:train loss 4.1796,val loss 4.1802
step 100:train loss 3.0096,val loss 3.0248
step 200:train loss 2.7213,val loss 2.6994
step 300:train loss 2.5970,val loss 2.6076
step 400:train loss 2.5359,val loss 2.5157
step 499:train loss 2.5142,val loss 2.4896


In [ ]:
# 关闭 Dropout
model.eval()

# 用编号为 0 的字符作为起始输入
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# 生成时不记录梯度
with torch.no_grad():
    result = model.generate(context, max_new_tokens=500)

# 取第一段序列并解码
generated_text = decode(result[0].tolist())
print(generated_text)

# 恢复训练模式，方便之后继续训练
model.train()


ruvoncoc,
Soms cingond t yt nd kilully sofalorlilln s d froutind bt Pofond
D zano netl int oul VEvhan m imame sowiteot h,d, t,
z f kunos t ther usorard flestte nost; t louer  fatlferurd t theanWad wifve y t, yonde E enor cang&rheuandost, dJur
Tun w I hoislne,
3 tureslld s td ishote

Thalld k d thighild coree af, mamareut pr hey,
ounc?

Annd orkamy mhide3 MEI
qwh: halovene b:
Acag, whrolrllvenculb ocen
ce,
Whsor soucinkce as.
Ces

HEF TAQO.
TI$:
Shad H:

A:
Sim;
I s ghowot wead or tto no ssvig t 


GPTLanguageModel(
  (token_embedding_table): Embedding(65, 64)
  (position_embedding_table): Embedding(32, 64)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=64, out_features=16, bias=False)
            (query): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedFoward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=Tr

In [ ]:
model.train()

for step in range(500):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if (step + 1) % 100 == 0:
        losses = estimate_loss()
        print(
            f"额外训练 {step + 1} 步："
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

额外训练 100 步：train loss 2.4012, val loss 2.4070
额外训练 200 步：train loss 2.3549, val loss 2.3540
额外训练 300 步：train loss 2.3305, val loss 2.3078
额外训练 400 步：train loss 2.2998, val loss 2.2958
额外训练 500 步：train loss 2.2555, val loss 2.2427


In [ ]:
# 关闭 Dropout
model.eval()

# 用编号为 0 的字符作为起始输入
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# 生成时不记录梯度
with torch.no_grad():
    result = model.generate(context, max_new_tokens=500)

# 取第一段序列并解码
generated_text = decode(result[0].tolist())
print(generated_text)

# 恢复训练模式，方便之后继续训练
model.train()


JENIHAUCALA: dand timhe aporef of youn is kepey, wiss so, ge cor le.


Whing.
TALEOLS:
He weye mat I to beghe yian four,

I ath
Maret ved diyst wirendeth wis gecht heasours:
Cin;
I bvitvenginke sha and but!

Inhy hat do thiy'
we whalay prey ke chenc; by heve icor hit's apard ment vis feellke is tisur heang bun:
Ou
Beme I make kgus o we now, wount proucgtiese ave tref the conene arne woul!

WIUMU
hish
Whay, beny
Sove, fur pliven and
Vendbly cassin a dont pecor,
Briry cand.


FEIWWISEERZANNS:
I so


GPTLanguageModel(
  (token_embedding_table): Embedding(65, 64)
  (position_embedding_table): Embedding(32, 64)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=64, out_features=16, bias=False)
            (query): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedFoward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=Tr